# CUDA WFR-ABF for the WCA dimer solvent

This notebook is a CUDA-oriented PyTorch version of `WCA_dimer.ipynb`. It is designed for Google Colab with a GPU runtime.

The CPU notebook uses NumPy and dense all-pairs arrays. This notebook instead:

- keeps simulation arrays as `torch.Tensor`s on the GPU;
- uses precomputed pair indices instead of full `(N,N)` pair matrices;
- avoids autograd and runs under `torch.inference_mode()`;
- batches all replicas in parallel;
- batches thermodynamic integration over chunks of reaction-coordinate values;
- transfers data back to CPU only for diagnostics and plotting.

In Colab, use **Runtime -> Change runtime type -> GPU** before running.


In [ ]:
import math
import time
from dataclasses import dataclass, replace

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. In Colab, set Runtime -> Change runtime type -> GPU.")

# Use CUDA in Colab. Fallbacks are only for local syntax/smoke testing.
def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = choose_device()
DTYPE = torch.float32
EPS = 1.0e-12

if DEVICE.type == "cuda":
    # Useful for Ampere/Lovelace GPUs. This notebook does not use matrix multiply
    # heavily, but enabling TF32 is harmless for ancillary kernels.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE)


## Configuration

The default physical parameters below match the original/book-scale notebook settings. The run flags are intentionally `False` so that the notebook can be opened and benchmarked without immediately starting a multi-hour computation.

Set these flags to run production calculations:

```python
RUN_MAIN_SIMULATION = True
RUN_TI_REFERENCE = True
```

The benchmark cell estimates runtime on the actual GPU before you commit to the full run.


In [ ]:
@dataclass(frozen=True)
class DimerWCAParams:
    n_dim: int = 10
    a: float = 1.5
    sigma: float = 1.0
    epsilon: float = 1.0
    h: float = 2.0
    w: float = 2.0
    beta: float = 1.0
    min_r: float = 0.65
    force_clip: float = 250.0

    @property
    def n_particles(self):
        return self.n_dim * self.n_dim

    @property
    def box_length(self):
        return self.n_dim * self.a

    @property
    def r0(self):
        return 2.0 ** (1.0 / 6.0) * self.sigma

    @property
    def wca_cutoff(self):
        return self.r0


@dataclass(frozen=True)
class SimConfig:
    n_replicas: int = 512
    dt: float = 2.0e-3
    n_steps: int = 200_000
    save_every: int = 2000
    seed: int = 42

    z_min: float = -0.2
    z_max: float = 1.2
    n_grid: int = 160

    abf_bandwidth: float = 0.040
    kde_bandwidth: float = 0.070
    fr_kernel_bandwidth: float = 0.060
    abf_smooth_sigma: float = 0.50

    mean_force_sample_clip: float = 500.0
    use_clipped_force_for_mean_force: bool = True
    abf_bias_scale: float = 1.0
    abf_edge_extrapolate: bool = True
    boundary_wall_strength: float = 80.0
    abf_force_clip: float = 40.0
    abf_warmup_steps: int = 10_000
    estimator_burn_in_steps: int = 10_000

    fr_rate: float = 0.25
    score_clip: float = 2.5
    fr_start_steps: int = 20_000
    fr_every: int = 1
    kernelized_fr_every: int = 10
    n_trace: int = 10
    marginal_burn_in_steps: int = 20_000

    barrier_center: float = 0.5
    transition_half_width: float = 0.15


@dataclass(frozen=True)
class TIConfig:
    z_min: float = -0.2
    z_max: float = 1.2
    n_z: int = 51
    n_replicas: int = 128
    dt: float = 2.0e-3
    n_thermalization: int = 5_000
    n_steps: int = 50_000
    sample_every: int = 20
    seed: int = 314159
    smooth_sigma: float = 1.0

    # Number of z values processed simultaneously on the GPU. Increase this on
    # large-memory GPUs; decrease if Colab reports out-of-memory.
    z_chunk_size: int = 8


params = DimerWCAParams()
sim = SimConfig()
ti = TIConfig(z_min=sim.z_min, z_max=sim.z_max, dt=sim.dt)

# Production defaults favor flatter sampling: sharper ABF, edge extrapolation, and a soft z-window wall.
RUN_MAIN_SIMULATION = True
RUN_TI_REFERENCE = False
RUN_BENCHMARK = False
RUN_TUNING_SWEEP = False
METHODS_TO_RUN = ("abf", "wfr_abf", "kwfr_abf")

print(params)
print(sim)
print(ti)
print(f"N={params.n_particles}, L={params.box_length:.3f}, r0={params.r0:.6f}")


## CUDA force engine

For WCA, only pair distances are needed. We precompute all unordered particle pairs except the dimer pair `(0,1)`. The force calculation then works on arrays of shape `(batch, n_pairs, 2)` instead of `(batch, N, N, 2)`. This reduces memory and avoids duplicate pair work.

The batch dimension represents independent replicas. In TI, it can represent several `z` values times several replicas per `z`.


In [ ]:
def as_tensor(x, device=DEVICE, dtype=DTYPE):
    return torch.as_tensor(x, device=device, dtype=dtype)


def wrap_positions(q, L):
    return torch.remainder(q, L)


def minimum_image(delta, L):
    return delta - L * torch.round(delta / L)


def clip_forces(forces, force_clip):
    norm = torch.linalg.norm(forces, dim=-1, keepdim=True)
    scale = torch.clamp(force_clip / torch.clamp(norm, min=EPS), max=1.0)
    return forces * scale


class WCADimerEngine:
    """GPU pair-list force engine for the dimer in WCA solvent."""

    def __init__(self, params, device=DEVICE, dtype=DTYPE):
        self.params = params
        self.device = device
        self.dtype = dtype
        self.N = params.n_particles
        self.L = float(params.box_length)
        self.sigma = float(params.sigma)
        self.epsilon = float(params.epsilon)
        self.h = float(params.h)
        self.w = float(params.w)
        self.r0 = float(params.r0)
        self.cutoff = float(params.wca_cutoff)
        self.min_r = float(params.min_r)

        pair_i, pair_j = torch.triu_indices(self.N, self.N, offset=1, device=device)
        keep = ~((pair_i == 0) & (pair_j == 1))
        self.pair_i = pair_i[keep].long()
        self.pair_j = pair_j[keep].long()
        self.n_pairs = int(self.pair_i.numel())

    def force(self, q, compute_energy=False):
        """Return physical forces -grad V, optionally with total energy."""
        B = q.shape[0]
        qi = q.index_select(1, self.pair_i)
        qj = q.index_select(1, self.pair_j)
        delta = minimum_image(qi - qj, self.L)
        r = torch.linalg.norm(delta, dim=-1)
        r_safe = torch.clamp(r, min=self.min_r * self.sigma)

        active = r <= self.cutoff
        inv = self.sigma / r_safe
        inv6 = inv**6
        inv12 = inv6**2
        dVdr = 4.0 * self.epsilon * (-12.0 * inv12 / r_safe + 6.0 * inv6 / r_safe)
        dVdr = torch.where(active, dVdr, torch.zeros_like(dVdr))
        f_pair = (-dVdr / torch.clamp(r_safe, min=EPS)).unsqueeze(-1) * delta

        forces = torch.zeros_like(q)
        idx_i = self.pair_i.view(1, -1, 1).expand(B, -1, 2)
        idx_j = self.pair_j.view(1, -1, 1).expand(B, -1, 2)
        forces.scatter_add_(1, idx_i, f_pair)
        forces.scatter_add_(1, idx_j, -f_pair)

        d01 = minimum_image(q[:, 0, :] - q[:, 1, :], self.L)
        r01 = torch.linalg.norm(d01, dim=1).clamp_min(EPS)
        u = (r01 - self.r0 - self.w) / self.w
        dVdr_dim = -4.0 * self.h * u * (1.0 - u**2) / self.w
        f01 = (-dVdr_dim / r01).unsqueeze(-1) * d01
        forces[:, 0, :] += f01
        forces[:, 1, :] -= f01

        if not compute_energy:
            return forces

        V_wca = 4.0 * self.epsilon * (inv12 - inv6) + self.epsilon
        V_wca = torch.where(active, V_wca, torch.zeros_like(V_wca)).sum(dim=1)
        V_dim = self.h * (1.0 - u**2) ** 2
        return V_wca + V_dim, forces


engine = WCADimerEngine(params, DEVICE, DTYPE)
print(f"precomputed WCA pairs: {engine.n_pairs} / dense {params.n_particles**2}")


## Geometry, local mean force, and initial conditions


In [ ]:
def dimer_displacement_and_length(q, params):
    d01 = minimum_image(q[:, 0, :] - q[:, 1, :], params.box_length)
    r01 = torch.linalg.norm(d01, dim=1).clamp_min(EPS)
    return d01, r01


def reaction_coordinate(q, params):
    _, r01 = dimer_displacement_and_length(q, params)
    return (r01 - params.r0) / (2.0 * params.w)


def grad_xi_dimer(q, params):
    d01, r01 = dimer_displacement_and_length(q, params)
    grad0 = d01 / (2.0 * params.w * r01[:, None])
    return grad0, -grad0


def local_mean_force(q, physical_forces, params):
    d01, r01 = dimer_displacement_and_length(q, params)
    grad_difference = physical_forces[:, 1, :] - physical_forces[:, 0, :]
    energetic = (params.w / r01) * torch.sum(d01 * grad_difference, dim=1)
    entropic = (2.0 * params.w) / (params.beta * r01)
    return energetic - entropic


def add_abf_force(q, forces, mean_force_at_z, params):
    grad0, grad1 = grad_xi_dimer(q, params)
    out = forces.clone()
    out[:, 0, :] += mean_force_at_z[:, None] * grad0
    out[:, 1, :] += mean_force_at_z[:, None] * grad1
    return out


def add_reaction_coordinate_wall_force(q, forces, z, sim, params):
    if sim.boundary_wall_strength <= 0.0:
        return forces
    upper_excess = torch.clamp(z - sim.z_max, min=0.0)
    lower_excess = torch.clamp(z - sim.z_min, max=0.0)
    dU_wall_dz = sim.boundary_wall_strength * (upper_excess + lower_excess)
    return add_abf_force(q, forces, -dU_wall_dz, params)


def lattice_initial_conditions(params, n_replicas, device=DEVICE, dtype=DTYPE, seed=0, jitter=0.015):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    coords = [((0.5 + i) * params.a, (0.5 + j) * params.a)
              for i in range(params.n_dim) for j in range(params.n_dim)]
    base = torch.tensor(coords, device=device, dtype=dtype)
    q = base.unsqueeze(0).repeat(n_replicas, 1, 1)
    shifts = torch.rand((n_replicas, 1, 2), device=device, dtype=dtype, generator=g) * params.box_length
    q = wrap_positions(q + shifts, params.box_length)
    if jitter > 0.0:
        q[:, 2:, :] = wrap_positions(
            q[:, 2:, :] + jitter * torch.randn(q[:, 2:, :].shape, device=device, dtype=dtype, generator=g),
            params.box_length,
        )
    q[:, 1, :] = wrap_positions(q[:, 0, :] + torch.tensor([0.0, params.r0], device=device, dtype=dtype), params.box_length)
    return q


def project_dimer_to_z(q, z, params):
    z = torch.as_tensor(z, device=q.device, dtype=q.dtype)
    if z.ndim == 0:
        z = z.expand(q.shape[0])
    target_r = params.r0 + 2.0 * params.w * z
    d01, r01 = dimer_displacement_and_length(q, params)
    direction = d01 / r01[:, None]
    midpoint = q[:, 1, :] + 0.5 * d01
    target_d = target_r[:, None] * direction
    out = q.clone()
    out[:, 0, :] = midpoint + 0.5 * target_d
    out[:, 1, :] = midpoint - 0.5 * target_d
    return wrap_positions(out, params.box_length)


## GPU ABF, KDE, and Fisher-Rao helpers

This version includes both simple Fisher-Rao (`wfr_abf`) and kernelized Fisher-Rao (`kwfr_abf`). The kernelized score follows `WFRABF.ipynb`, using `fr_kernel_bandwidth` for the KDE and pair kernel term. It is O(R^2), so it has its own update interval `kernelized_fr_every`.


In [ ]:
def gaussian_kernel_torch(diff, bandwidth):
    bw = float(max(bandwidth, EPS))
    return torch.exp(-0.5 * (diff / bw) ** 2) / (bw * math.sqrt(2.0 * math.pi))


def smooth_profile_torch(y, sigma_grid_points):
    sigma = float(sigma_grid_points)
    if sigma <= 0:
        return y.clone()
    radius = max(1, int(math.ceil(4.0 * sigma)))
    x = torch.arange(-radius, radius + 1, device=y.device, dtype=y.dtype)
    kernel = torch.exp(-0.5 * (x / sigma) ** 2)
    kernel = kernel / kernel.sum()
    yp = F.pad(y.view(1, 1, -1), (radius, radius), mode="replicate")
    return F.conv1d(yp, kernel.view(1, 1, -1)).view(-1)


def cumulative_trapezoid_torch(y, x):
    out = torch.zeros_like(y)
    out[1:] = torch.cumsum(0.5 * (y[1:] + y[:-1]) * (x[1:] - x[:-1]), dim=0)
    return out


def normalize_profile_zero_at_midpoint_torch(profile, grid, midpoint=0.5):
    idx = torch.argmin(torch.abs(grid - midpoint))
    return profile - profile[idx]


def trapz_torch(y, x):
    return torch.sum(0.5 * (y[1:] + y[:-1]) * (x[1:] - x[:-1]))


def normalize_density_on_grid_torch(p_grid, grid):
    mass = trapz_torch(p_grid, grid)
    return p_grid / torch.clamp(mass, min=EPS)


def kde_1d_torch(eval_points, samples, bandwidth, z_min, z_max):
    reflected = torch.cat([samples, 2.0 * z_min - samples, 2.0 * z_max - samples])
    diff = eval_points[:, None] - reflected[None, :]
    p = gaussian_kernel_torch(diff, bandwidth).sum(dim=1) / max(samples.numel(), 1)
    return torch.clamp(p, min=EPS)


def free_energy_from_density_torch(p_grid, grid, beta):
    Fz = -(1.0 / beta) * torch.log(torch.clamp(p_grid, min=EPS))
    return normalize_profile_zero_at_midpoint_torch(Fz, grid)


def mean_force_from_free_energy_torch(free_energy, grid):
    # torch.gradient is available, but this manual form is backend-stable.
    out = torch.empty_like(free_energy)
    out[1:-1] = (free_energy[2:] - free_energy[:-2]) / (grid[2:] - grid[:-2])
    out[0] = (free_energy[1] - free_energy[0]) / (grid[1] - grid[0])
    out[-1] = (free_energy[-1] - free_energy[-2]) / (grid[-1] - grid[-2])
    return out


def interp_uniform_grid(profile, grid, z, outside_value=0.0):
    dz = grid[1] - grid[0]
    x = (z - grid[0]) / dz
    idx0 = torch.floor(x).long()
    inside = (idx0 >= 0) & (idx0 < grid.numel() - 1)
    idx0c = idx0.clamp(0, grid.numel() - 2)
    frac = (x - idx0c.to(z.dtype)).clamp(0.0, 1.0)
    val = (1.0 - frac) * profile[idx0c] + frac * profile[idx0c + 1]
    return torch.where(inside, val, torch.full_like(val, outside_value))


def interp_uniform_grid_edge(profile, grid, z):
    dz = grid[1] - grid[0]
    x = (z - grid[0]) / dz
    idx0 = torch.floor(x).long()
    idx0c = idx0.clamp(0, grid.numel() - 2)
    frac = (x - idx0c.to(z.dtype)).clamp(0.0, 1.0)
    val = (1.0 - frac) * profile[idx0c] + frac * profile[idx0c + 1]
    val = torch.where(z < grid[0], profile[0].expand_as(val), val)
    val = torch.where(z > grid[-1], profile[-1].expand_as(val), val)
    return val


class TorchKernelABFEstimator:
    def __init__(self, z_grid, bandwidth, smooth_sigma=0.0, edge_extrapolate=False):
        self.z_grid = z_grid
        self.bandwidth = float(bandwidth)
        self.smooth_sigma = float(smooth_sigma)
        self.edge_extrapolate = bool(edge_extrapolate)
        self.num = torch.zeros_like(z_grid)
        self.den = torch.zeros_like(z_grid)
        self.n_updates = 0

    def update(self, z_samples, force_samples):
        weights = gaussian_kernel_torch(self.z_grid[:, None] - z_samples[None, :], self.bandwidth)
        self.num += torch.sum(weights * force_samples[None, :], dim=1)
        self.den += torch.sum(weights, dim=1)
        self.n_updates += int(z_samples.numel())

    def mean_force_profile(self):
        raw = self.num / torch.clamp(self.den, min=EPS)
        raw = torch.where(self.den > EPS, raw, torch.zeros_like(raw))
        return smooth_profile_torch(raw, self.smooth_sigma)

    def evaluate(self, z_samples):
        profile = self.mean_force_profile()
        if self.edge_extrapolate:
            return interp_uniform_grid_edge(profile, self.z_grid, z_samples)
        return interp_uniform_grid(profile, self.z_grid, z_samples, outside_value=0.0)

    def pmf_profile(self):
        pmf = cumulative_trapezoid_torch(self.mean_force_profile(), self.z_grid)
        return normalize_profile_zero_at_midpoint_torch(pmf, self.z_grid)


def recentered_clipped_score_torch(raw_score, score_clip):
    score = raw_score - raw_score.mean()
    for _ in range(3):
        score = torch.clamp(score, -score_clip, score_clip)
        score = score - score.mean()
    return torch.clamp(score, -score_clip, score_clip)


def fisher_rao_score_torch(z_samples, sim):
    p_at = kde_1d_torch(z_samples, z_samples, sim.kde_bandwidth, sim.z_min, sim.z_max)
    inside = (z_samples >= sim.z_min) & (z_samples <= sim.z_max)
    u = torch.where(inside, torch.full_like(z_samples, 1.0 / (sim.z_max - sim.z_min)), torch.full_like(z_samples, EPS))
    log_ratio = torch.log(torch.clamp(p_at, min=EPS)) - torch.log(torch.clamp(u, min=EPS))
    return recentered_clipped_score_torch(log_ratio, sim.score_clip)


def fisher_rao_kernelized_score_torch(z_samples, grid, sim):
    bandwidth = sim.fr_kernel_bandwidth
    p_at = kde_1d_torch(z_samples, z_samples, bandwidth, sim.z_min, sim.z_max)
    p_grid = normalize_density_on_grid_torch(
        kde_1d_torch(grid, z_samples, bandwidth, sim.z_min, sim.z_max),
        grid,
    )

    inside_particles = (z_samples >= sim.z_min) & (z_samples <= sim.z_max)
    u_at = torch.where(
        inside_particles,
        torch.full_like(z_samples, 1.0 / (sim.z_max - sim.z_min)),
        torch.full_like(z_samples, EPS),
    )
    inside_grid = (grid >= sim.z_min) & (grid <= sim.z_max)
    u_grid = torch.where(
        inside_grid,
        torch.full_like(grid, 1.0 / (sim.z_max - sim.z_min)),
        torch.full_like(grid, EPS),
    )

    log_ratio_at = torch.log(torch.clamp(p_at, min=EPS)) - torch.log(torch.clamp(u_at, min=EPS))
    grid_log_ratio = torch.log(torch.clamp(p_grid, min=EPS)) - torch.log(torch.clamp(u_grid, min=EPS))
    grid_baseline = trapz_torch(p_grid * grid_log_ratio, grid)

    diff = z_samples[:, None] - z_samples[None, :]
    kernel_matrix = gaussian_kernel_torch(diff, bandwidth)
    kernel_term = torch.mean(kernel_matrix / torch.clamp(p_at[None, :], min=EPS), dim=1)
    raw_score = log_ratio_at + kernel_term - grid_baseline - 1.0
    return recentered_clipped_score_torch(raw_score, sim.score_clip)


def fixed_population_birth_death_torch(q, score, sim, fr_interval=None):
    R = q.shape[0]
    score = recentered_clipped_score_torch(score, sim.score_clip)
    interval = sim.fr_every if fr_interval is None else fr_interval
    dt_eff = sim.dt * max(int(interval), 1)

    death_weights = torch.clamp(score, min=0.0)
    birth_weights = torch.clamp(-score, min=0.0)
    death_mass = torch.sum(death_weights)
    birth_mass = torch.sum(birth_weights)
    if bool(((death_mass <= EPS) | (birth_mass <= EPS)).detach().cpu().item()):
        stats = {
            "replacement": 0,
            "mean_abs_score": float(torch.mean(torch.abs(score)).detach().cpu().item()),
            "score_mean": float(torch.mean(score).detach().cpu().item()),
        }
        return q.clone(), stats

    death_prob = torch.where(
        death_weights > 0.0,
        1.0 - torch.exp(-sim.fr_rate * death_weights * dt_eff),
        torch.zeros_like(death_weights),
    )
    death_indices = torch.nonzero(torch.rand(R, device=q.device, dtype=q.dtype) < death_prob, as_tuple=False).flatten()
    n_events = int(death_indices.numel())
    if n_events == 0:
        stats = {
            "replacement": 0,
            "mean_abs_score": float(torch.mean(torch.abs(score)).detach().cpu().item()),
            "score_mean": float(torch.mean(score).detach().cpu().item()),
        }
        return q.clone(), stats

    clone_sources = torch.multinomial(birth_weights, n_events, replacement=True)
    q_new = q.clone()
    q_new[death_indices] = q.index_select(0, clone_sources)
    stats = {
        "replacement": int(n_events),
        "mean_abs_score": float(torch.mean(torch.abs(score)).detach().cpu().item()),
        "score_mean": float(torch.mean(score).detach().cpu().item()),
    }
    return q_new, stats


## Diagnostics


In [ ]:
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def l2_error_to_uniform_torch(p_grid, grid, sim):
    u = torch.full_like(grid, 1.0 / (sim.z_max - sim.z_min))
    return torch.sqrt(trapz_torch((p_grid - u) ** 2, grid)).item()


def transition_thresholds(sim):
    return sim.barrier_center - sim.transition_half_width, sim.barrier_center + sim.transition_half_width


def classify_states_torch(z, sim):
    low, high = transition_thresholds(sim)
    compact = torch.mean((z < low).to(torch.float32)).item()
    stretched = torch.mean((z > high).to(torch.float32)).item()
    transition = 1.0 - compact - stretched
    return compact, transition, stretched


def profile_l2_error_np(profile, reference, grid):
    return math.sqrt(np.trapezoid((np.asarray(profile) - np.asarray(reference)) ** 2, grid) / (grid[-1] - grid[0]))


## Main GPU samplers


In [ ]:
@torch.inference_mode()
def run_sampler_gpu(method, params, sim, engine, initial_q=None, verbose=True):
    if method not in {"unbiased", "abf", "wfr_abf", "kwfr_abf"}:
        raise ValueError("method must be 'unbiased', 'abf', 'wfr_abf', or 'kwfr_abf'")
    torch.manual_seed(sim.seed)
    q = lattice_initial_conditions(params, sim.n_replicas, engine.device, engine.dtype, seed=sim.seed) if initial_q is None else initial_q.clone()
    grid = torch.linspace(sim.z_min, sim.z_max, sim.n_grid, device=engine.device, dtype=engine.dtype)
    bias_estimator = TorchKernelABFEstimator(grid, sim.abf_bandwidth, sim.abf_smooth_sigma, edge_extrapolate=sim.abf_edge_extrapolate)
    production_estimator = TorchKernelABFEstimator(grid, sim.abf_bandwidth, sim.abf_smooth_sigma, edge_extrapolate=sim.abf_edge_extrapolate)
    noise_scale = math.sqrt(2.0 * sim.dt / params.beta)
    total_replacement_events = 0
    marginal_p_sum = torch.zeros_like(grid)
    marginal_p_count = 0

    diag = {k: [] for k in [
        "steps", "times", "p_grid", "sampled_p_grid", "l2_marginal", "sampled_l2_marginal", "mean_force", "pmf",
        "hist_free_energy", "hist_mean_force", "state_fractions", "trace_z",
        "replacement_events", "mean_abs_score", "score_mean",
    ]}

    t0 = time.perf_counter()
    for step in range(sim.n_steps + 1):
        forces_raw = engine.force(q, compute_energy=False)
        forces_physical = clip_forces(forces_raw, params.force_clip)
        z = reaction_coordinate(q, params)

        if method in {"abf", "wfr_abf", "kwfr_abf"}:
            mean_force_input = forces_physical if sim.use_clipped_force_for_mean_force else forces_raw
            f_local = local_mean_force(q, mean_force_input, params)
            f_local = torch.clamp(f_local, -sim.mean_force_sample_clip, sim.mean_force_sample_clip)
            bias_estimator.update(z, f_local)
            if step >= sim.estimator_burn_in_steps:
                production_estimator.update(z, f_local)
            ramp = min(1.0, step / max(sim.abf_warmup_steps, 1))
            abf_at_z = sim.abf_bias_scale * ramp * torch.clamp(bias_estimator.evaluate(z), -sim.abf_force_clip, sim.abf_force_clip)
        else:
            abf_at_z = torch.zeros_like(z)

        transport = forces_physical
        if method in {"abf", "wfr_abf", "kwfr_abf"}:
            transport = clip_forces(add_abf_force(q, transport, abf_at_z, params), params.force_clip)
        transport = clip_forces(add_reaction_coordinate_wall_force(q, transport, z, sim, params), params.force_clip)

        if step % sim.save_every == 0 or step == sim.n_steps:
            report_estimator = production_estimator if production_estimator.n_updates > 0 else bias_estimator
            p_grid = normalize_density_on_grid_torch(kde_1d_torch(grid, z, sim.kde_bandwidth, sim.z_min, sim.z_max), grid)
            if step >= sim.marginal_burn_in_steps:
                marginal_p_sum += p_grid
                marginal_p_count += 1
            sampled_p_grid = marginal_p_sum / max(marginal_p_count, 1) if marginal_p_count else p_grid
            hist_fe = free_energy_from_density_torch(p_grid, grid, params.beta)
            hist_mf = mean_force_from_free_energy_torch(hist_fe, grid)
            diag["steps"].append(step)
            diag["times"].append(step * sim.dt)
            diag["p_grid"].append(to_numpy(p_grid))
            diag["sampled_p_grid"].append(to_numpy(sampled_p_grid))
            diag["l2_marginal"].append(l2_error_to_uniform_torch(p_grid, grid, sim))
            diag["sampled_l2_marginal"].append(l2_error_to_uniform_torch(sampled_p_grid, grid, sim))
            diag["mean_force"].append(to_numpy(report_estimator.mean_force_profile()))
            diag["pmf"].append(to_numpy(report_estimator.pmf_profile()))
            diag["hist_free_energy"].append(to_numpy(hist_fe))
            diag["hist_mean_force"].append(to_numpy(hist_mf))
            diag["state_fractions"].append(classify_states_torch(z, sim))
            diag["trace_z"].append(to_numpy(z[:sim.n_trace]))

        if step == sim.n_steps:
            break

        q = wrap_positions(q + sim.dt * transport + noise_scale * torch.randn_like(q), params.box_length)
        if method in {"wfr_abf", "kwfr_abf"}:
            next_step = step + 1
            fr_interval = sim.kernelized_fr_every if method == "kwfr_abf" else sim.fr_every
            do_fr = (
                next_step >= sim.fr_start_steps
                and (next_step - sim.fr_start_steps) % max(int(fr_interval), 1) == 0
            )
            if do_fr:
                z_new = reaction_coordinate(q, params)
                if method == "kwfr_abf":
                    score = fisher_rao_kernelized_score_torch(z_new, grid, sim)
                else:
                    score = fisher_rao_score_torch(z_new, sim)
                q, stats = fixed_population_birth_death_torch(q, score, sim, fr_interval=fr_interval)
                total_replacement_events += stats["replacement"]
                diag["replacement_events"].append(stats["replacement"])
                diag["mean_abs_score"].append(stats["mean_abs_score"])
                diag["score_mean"].append(stats["score_mean"])
            else:
                diag["replacement_events"].append(0)
                diag["mean_abs_score"].append(0.0)
                diag["score_mean"].append(0.0)
        else:
            diag["replacement_events"].append(0)
            diag["mean_abs_score"].append(0.0)
            diag["score_mean"].append(0.0)

    diag["runtime_seconds"] = time.perf_counter() - t0
    diag["method"] = method
    diag["grid"] = to_numpy(grid)
    diag["q_final"] = q.detach().cpu()
    diag["total_replacement_events"] = total_replacement_events
    for key in [
        "steps", "times", "p_grid", "sampled_p_grid", "l2_marginal", "sampled_l2_marginal", "mean_force", "pmf",
        "hist_free_energy", "hist_mean_force", "state_fractions", "trace_z",
    ]:
        diag[key] = np.asarray(diag[key])
    if verbose:
        c, tr, s = diag["state_fractions"][-1]
        extra = ""
        if method in {"wfr_abf", "kwfr_abf"}:
            extra = f", replacements {total_replacement_events}"
        print(f"{method:9s}: {diag['runtime_seconds']:.1f}s, final marginal L2 {diag['l2_marginal'][-1]:.4f}, sampled L2 {diag['sampled_l2_marginal'][-1]:.4f}, C/T/S {c:.2f}/{tr:.2f}/{s:.2f}{extra}")
    return diag


def run_all_methods_gpu(params, sim, engine, methods=METHODS_TO_RUN):
    initial_q = lattice_initial_conditions(params, sim.n_replicas, engine.device, engine.dtype, seed=sim.seed)
    results = {}
    for method in methods:
        results[method] = run_sampler_gpu(method, params, sim, engine, initial_q=initial_q, verbose=True)
    return results


## CUDA thermodynamic integration reference

The TI computation is batched over chunks of `z` values. For example, with `z_chunk_size=8` and `n_replicas=64`, one GPU batch contains `512` constrained systems. Decrease `z_chunk_size` if Colab runs out of memory.


In [ ]:
@torch.inference_mode()
def constrained_ti_reference_gpu(params, sim, ti, engine, eval_grid_np=None, verbose=True):
    torch.manual_seed(ti.seed)
    z_all = torch.linspace(ti.z_min, ti.z_max, ti.n_z, device=engine.device, dtype=engine.dtype)
    mf = torch.zeros(ti.n_z, device=engine.device, dtype=engine.dtype)
    mf2 = torch.zeros(ti.n_z, device=engine.device, dtype=engine.dtype)
    count = torch.zeros(ti.n_z, device=engine.device, dtype=engine.dtype)
    noise_scale = math.sqrt(2.0 * ti.dt / params.beta)

    t0 = time.perf_counter()
    for start in range(0, ti.n_z, ti.z_chunk_size):
        end = min(start + ti.z_chunk_size, ti.n_z)
        z_chunk = z_all[start:end]
        C = z_chunk.numel()
        q0 = lattice_initial_conditions(params, C * ti.n_replicas, engine.device, engine.dtype, seed=ti.seed + start)
        z_batch = z_chunk.repeat_interleave(ti.n_replicas)
        q = project_dimer_to_z(q0, z_batch, params)

        total_steps = ti.n_thermalization + ti.n_steps
        for step in range(total_steps):
            forces_raw = engine.force(q, compute_energy=False)
            transport = clip_forces(forces_raw, params.force_clip)
            q = wrap_positions(q + ti.dt * transport + noise_scale * torch.randn_like(q), params.box_length)
            q = project_dimer_to_z(q, z_batch, params)

            if step >= ti.n_thermalization and (step - ti.n_thermalization) % ti.sample_every == 0:
                sample_forces = engine.force(q, compute_energy=False)
                if sim.use_clipped_force_for_mean_force:
                    sample_forces = clip_forces(sample_forces, params.force_clip)
                f_local = local_mean_force(q, sample_forces, params).view(C, ti.n_replicas)
                mf[start:end] += f_local.sum(dim=1)
                mf2[start:end] += (f_local**2).sum(dim=1)
                count[start:end] += ti.n_replicas

        if verbose:
            elapsed = time.perf_counter() - t0
            chunk_count = torch.clamp(count[start:end], min=1.0)
            chunk_mean = mf[start:end] / chunk_count
            chunk_var = torch.clamp(
                (mf2[start:end] - chunk_count * chunk_mean**2)
                / torch.clamp(chunk_count - 1.0, min=1.0),
                min=0.0,
            )
            chunk_std = torch.sqrt(chunk_var)
            chunk_stderr = chunk_std / torch.sqrt(chunk_count)
            for local_i in range(C):
                global_i = start + local_i
                print(
                    f"TI {global_i + 1:02d}/{ti.n_z}: "
                    f"z={float(z_all[global_i].detach().cpu()): .3f}, "
                    f"F'={float(chunk_mean[local_i].detach().cpu()): .4f} +/- "
                    f"{float(chunk_stderr[local_i].detach().cpu()):.4f} "
                    f"(std={float(chunk_std[local_i].detach().cpu()):.4f}, "
                    f"samples={int(count[global_i].detach().cpu().item())})"
                )
            print(f"TI chunk {start:02d}:{end:02d} done, elapsed {elapsed/60:.1f} min")

    mean_force = mf / torch.clamp(count, min=1.0)
    variance = torch.clamp(
        (mf2 - torch.clamp(count, min=1.0) * mean_force**2)
        / torch.clamp(count - 1.0, min=1.0),
        min=0.0,
    )
    sample_std = torch.sqrt(variance)
    stderr = sample_std / torch.sqrt(torch.clamp(count, min=1.0))
    mean_force_smooth = smooth_profile_torch(mean_force, ti.smooth_sigma)
    free_energy = normalize_profile_zero_at_midpoint_torch(cumulative_trapezoid_torch(mean_force_smooth, z_all), z_all)

    if eval_grid_np is None:
        eval_grid = torch.linspace(sim.z_min, sim.z_max, sim.n_grid, device=engine.device, dtype=engine.dtype)
    else:
        eval_grid = torch.as_tensor(eval_grid_np, device=engine.device, dtype=engine.dtype)
    mf_eval = interp_uniform_grid(mean_force_smooth, z_all, eval_grid, outside_value=0.0)
    fe_eval = interp_uniform_grid(free_energy, z_all, eval_grid, outside_value=float(free_energy[0].item()))
    fe_eval = normalize_profile_zero_at_midpoint_torch(fe_eval, eval_grid)

    return {
        "source": "thermodynamic_integration_cuda",
        "label": "CUDA thermodynamic integration reference",
        "grid": to_numpy(eval_grid),
        "mean_force": to_numpy(mf_eval),
        "free_energy": to_numpy(fe_eval),
        "z_ti": to_numpy(z_all),
        "mean_force_ti_raw": to_numpy(mean_force),
        "mean_force_ti_smooth": to_numpy(mean_force_smooth),
        "free_energy_ti": to_numpy(free_energy),
        "std": to_numpy(sample_std),
        "stderr": to_numpy(stderr),
        "n_samples": to_numpy(count),
        "runtime_seconds": time.perf_counter() - t0,
        "config": ti,
    }


## Runtime benchmark on the current GPU

Run this before production. It estimates the main simulation and TI reference time for the current device and configs.


In [ ]:
@torch.inference_mode()
def benchmark_gpu_runtime(params, sim, ti, engine, n_bench=200):
    torch.manual_seed(sim.seed)
    noise_scale = math.sqrt(2.0 * sim.dt / params.beta)

    def bench_step(batch_replicas, constrained=False):
        q = lattice_initial_conditions(params, batch_replicas, engine.device, engine.dtype, seed=123)
        if constrained:
            z = torch.full((batch_replicas,), 0.5, device=engine.device, dtype=engine.dtype)
            q = project_dimer_to_z(q, z, params)
        for _ in range(20):
            f = engine.force(q, compute_energy=False)
            q = wrap_positions(q + sim.dt * clip_forces(f, params.force_clip) + noise_scale * torch.randn_like(q), params.box_length)
            if constrained:
                q = project_dimer_to_z(q, z, params)
        if engine.device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_bench):
            f = engine.force(q, compute_energy=False)
            q = wrap_positions(q + sim.dt * clip_forces(f, params.force_clip) + noise_scale * torch.randn_like(q), params.box_length)
            if constrained:
                q = project_dimer_to_z(q, z, params)
        if engine.device.type == "cuda":
            torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_bench

    main_step = bench_step(sim.n_replicas, constrained=False)
    ti_batch = min(ti.z_chunk_size, ti.n_z) * ti.n_replicas
    ti_step = bench_step(ti_batch, constrained=True)

    # Main methods have ABF/KDE overhead; add a conservative multiplier.
    main_total = sim.n_steps * main_step * 3.4
    n_chunks = math.ceil(ti.n_z / ti.z_chunk_size)
    samples_per_chunk = ti.n_steps // ti.sample_every
    ti_total = n_chunks * ((ti.n_thermalization + ti.n_steps) * ti_step + samples_per_chunk * ti_step)

    print(f"device: {engine.device}")
    print(f"main batch replicas: {sim.n_replicas}, measured base step: {main_step:.6f}s")
    print(f"TI batch systems: {ti_batch}, measured constrained step: {ti_step:.6f}s")
    print(f"estimated selected-method simulation: {main_total/3600:.2f} h")
    print(f"estimated TI reference: {ti_total/3600:.2f} h")
    print(f"estimated total: {(main_total + ti_total)/3600:.2f} h")
    return {"main_step": main_step, "ti_step": ti_step, "main_total": main_total, "ti_total": ti_total}


if RUN_BENCHMARK:
    benchmark = benchmark_gpu_runtime(params, sim, ti, engine, n_bench=200)


## Run production calculations

Set the flags in the configuration cell before running these cells.


In [ ]:
if RUN_MAIN_SIMULATION:
    results = run_all_methods_gpu(params, sim, engine)
else:
    results = None
    print("RUN_MAIN_SIMULATION=False; skipping main simulation.")


In [ ]:
if RUN_TI_REFERENCE:
    if results is None:
        eval_grid_np = np.linspace(sim.z_min, sim.z_max, sim.n_grid)
    else:
        first_result = next(iter(results.values()))
        eval_grid_np = first_result["grid"]
    ti_reference = constrained_ti_reference_gpu(params, sim, ti, engine, eval_grid_np=eval_grid_np, verbose=True)
    print(f"TI runtime: {ti_reference['runtime_seconds']/3600:.2f} h")
else:
    ti_reference = globals().get("ti_reference", None)
    if ti_reference is None:
        print("RUN_TI_REFERENCE=False; no existing TI reference is available.")
    else:
        print("RUN_TI_REFERENCE=False; reusing existing TI reference from this session.")


## Short hyperparameter sweep

Use this before committing to another full run. The sweep now tests sharper ABF bandwidths, ABF force scaling, and kernelized Fisher-Rao bandwidths with the same edge-extrapolated bias and soft reaction-coordinate wall used by production.


In [ ]:
TUNING_COMMON_OVERRIDES = dict(
    n_replicas=256,
    n_steps=40_000,
    save_every=2_000,
    abf_warmup_steps=5_000,
    estimator_burn_in_steps=5_000,
    fr_start_steps=8_000,
    n_trace=4,
)

TUNING_CANDIDATES = [
    dict(name="abf_bw004", method="abf", overrides=dict(abf_bandwidth=0.040, abf_smooth_sigma=0.50, abf_bias_scale=1.0)),
    dict(name="abf_bw004_scale075", method="abf", overrides=dict(abf_bandwidth=0.040, abf_smooth_sigma=0.50, abf_bias_scale=0.75)),
    dict(name="abf_bw004_scale125", method="abf", overrides=dict(abf_bandwidth=0.040, abf_smooth_sigma=0.50, abf_bias_scale=1.25)),
    dict(name="wfr_wall", method="wfr_abf", overrides=dict(fr_rate=0.25, score_clip=2.5, kde_bandwidth=0.070)),
    dict(name="kwfr_bw005", method="kwfr_abf", overrides=dict(fr_rate=0.25, score_clip=2.5, fr_kernel_bandwidth=0.050, kernelized_fr_every=10)),
    dict(name="kwfr_bw006", method="kwfr_abf", overrides=dict(fr_rate=0.25, score_clip=2.5, fr_kernel_bandwidth=0.060, kernelized_fr_every=10)),
    dict(name="kwfr_bw007", method="kwfr_abf", overrides=dict(fr_rate=0.25, score_clip=2.5, fr_kernel_bandwidth=0.070, kernelized_fr_every=10)),
    dict(name="kwfr_bw006_fast", method="kwfr_abf", overrides=dict(fr_rate=0.32, score_clip=2.5, fr_kernel_bandwidth=0.060, kernelized_fr_every=10)),
]

SELECTED_TUNED_CANDIDATE = "kwfr_bw006"



def _merged_sim(base_sim, common_overrides, candidate_overrides):
    overrides = dict(common_overrides)
    overrides.update(candidate_overrides)
    return replace(base_sim, **overrides)


def _profile_l2_on_reference_grid(profile, profile_grid, reference_profile, reference_grid):
    profile_eval = np.interp(reference_grid, profile_grid, profile)
    return profile_l2_error_np(profile_eval, reference_profile, reference_grid)


def _final_reference_errors(res, reference):
    if reference is None:
        return math.nan, math.nan
    if res["method"] == "unbiased":
        mf = res["hist_mean_force"][-1]
        fe = res["hist_free_energy"][-1]
    else:
        mf = res["mean_force"][-1]
        fe = res["pmf"][-1]
    ref_grid = np.asarray(reference["grid"])
    return (
        _profile_l2_on_reference_grid(mf, res["grid"], reference["mean_force"], ref_grid),
        _profile_l2_on_reference_grid(fe, res["grid"], reference["free_energy"], ref_grid),
    )


def run_hyperparameter_tuning_gpu(params, base_sim, engine, candidates=TUNING_CANDIDATES, common_overrides=TUNING_COMMON_OVERRIDES, reference=None):
    rows = []
    for candidate in candidates:
        candidate_sim = _merged_sim(base_sim, common_overrides, candidate.get("overrides", {}))
        method = candidate["method"]
        initial_q = lattice_initial_conditions(params, candidate_sim.n_replicas, engine.device, engine.dtype, seed=candidate_sim.seed)
        res = run_sampler_gpu(method, params, candidate_sim, engine, initial_q=initial_q, verbose=False)
        c, tr, s = res["state_fractions"][-1]
        l2_mf, l2_fe = _final_reference_errors(res, reference)
        row = dict(
            name=candidate["name"],
            method=method,
            l2_marginal=float(res["l2_marginal"][-1]),
            sampled_l2_marginal=float(res["sampled_l2_marginal"][-1]),
            l2_mean_force_ref=float(l2_mf),
            l2_free_energy_ref=float(l2_fe),
            compact=float(c),
            transition=float(tr),
            stretched=float(s),
            replacements=int(res.get("total_replacement_events", 0)),
            runtime_seconds=float(res["runtime_seconds"]),
            sim=candidate_sim,
            result=res,
        )
        rows.append(row)
        print(
            f"{row['name']:18s} {method:9s} "
            f"marginal={row['l2_marginal']:.4f} sampled={row['sampled_l2_marginal']:.4f} "
            f"F'={row['l2_mean_force_ref']:.3f} F={row['l2_free_energy_ref']:.3f} "
            f"C/T/S={c:.2f}/{tr:.2f}/{s:.2f} "
            f"repl={row['replacements']} runtime={row['runtime_seconds']:.1f}s"
        )
        if engine.device.type == "cuda":
            torch.cuda.empty_cache()

    sort_key = "l2_free_energy_ref" if reference is not None else "l2_marginal"
    rows_sorted = sorted(rows, key=lambda r: math.inf if math.isnan(r[sort_key]) else r[sort_key])
    print(f"\nSorted by {sort_key}:")
    for row in rows_sorted:
        print(
            f"{row['name']:18s} {row['method']:9s} "
            f"marginal={row['l2_marginal']:.4f} sampled={row['sampled_l2_marginal']:.4f} "
            f"F'={row['l2_mean_force_ref']:.3f} F={row['l2_free_energy_ref']:.3f} "
            f"repl={row['replacements']}"
        )
    return rows


if RUN_TUNING_SWEEP:
    tuning_reference = ti_reference if 'ti_reference' in globals() and ti_reference is not None else None
    tuning_results = run_hyperparameter_tuning_gpu(params, sim, engine, reference=tuning_reference)
else:
    tuning_results = None
    print("RUN_TUNING_SWEEP=False; skipping short hyperparameter sweep.")


## Plotting and error diagnostics


In [ ]:
METHOD_ORDER = list(METHODS_TO_RUN)
METHOD_LABELS = {"unbiased": "Unbiased", "abf": "ABF", "wfr_abf": "WFR-ABF", "kwfr_abf": "Kernel WFR-ABF"}
METHOD_COLORS = {"unbiased": "tab:gray", "abf": "tab:blue", "wfr_abf": "tab:orange", "kwfr_abf": "tab:green"}


def estimated_profile_series(res):
    if res["method"] == "unbiased":
        return res["hist_mean_force"], res["hist_free_energy"]
    return res["mean_force"], res["pmf"]


def attach_reference_errors(results, reference):
    ref_grid = reference["grid"]
    for key, res in results.items():
        mf_series, fe_series = estimated_profile_series(res)
        res["l2_mean_force_ref"] = np.array([profile_l2_error_np(p, reference["mean_force"], ref_grid) for p in mf_series])
        res["l2_free_energy_ref"] = np.array([profile_l2_error_np(p, reference["free_energy"], ref_grid) for p in fe_series])


def plot_method_diagnostics(results, method, reference, sim):
    res = results[method]
    grid = res["grid"]
    mf_series, fe_series = estimated_profile_series(res)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    color = METHOD_COLORS[method]
    label = METHOD_LABELS[method]
    axes[0].plot(reference["grid"], reference["mean_force"], color="black", lw=2, label="TI reference")
    axes[0].plot(grid, mf_series[-1], color=color, lw=2, label=label)
    axes[0].axhline(0, color="0.7", lw=0.8)
    axes[0].set_title("Mean force")
    axes[0].set_xlabel("z")
    axes[0].set_ylabel(r"$F'(z)$")
    axes[0].legend(frameon=False)

    axes[1].plot(reference["grid"], reference["free_energy"], color="black", lw=2, label="TI reference")
    axes[1].plot(grid, fe_series[-1], color=color, lw=2, label=label)
    axes[1].axhline(0, color="0.7", lw=0.8)
    axes[1].set_title("Free energy")
    axes[1].set_xlabel("z")
    axes[1].set_ylabel(r"$F(z)$, shifted")
    axes[1].legend(frameon=False)

    axes[2].plot(grid, res["p_grid"][-1], color=color, lw=1.5, alpha=0.55, label=f"{label} final")
    axes[2].plot(grid, res["sampled_p_grid"][-1], color=color, lw=2.2, label=f"{label} sampled")
    axes[2].plot(grid, np.full_like(grid, 1.0 / (sim.z_max - sim.z_min)), color="black", ls="--", lw=1.5, label="uniform")
    axes[2].set_title("Reaction-coordinate marginal")
    axes[2].set_xlabel("z")
    axes[2].set_ylabel("density")
    axes[2].legend(frameon=False)
    fig.suptitle(f"{label}: final diagnostics", y=1.03)
    plt.tight_layout()
    plt.show()


def plot_l2_errors(results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for key in METHOD_ORDER:
        res = results[key]
        axes[0].plot(res["times"], res["l2_mean_force_ref"], color=METHOD_COLORS[key], lw=2, label=METHOD_LABELS[key])
        axes[1].plot(res["times"], res["l2_free_energy_ref"], color=METHOD_COLORS[key], lw=2, label=METHOD_LABELS[key])
    axes[0].set_title("Mean-force error")
    axes[0].set_xlabel("time")
    axes[0].set_ylabel(r"$\|\widehat F' - F'_{TI}\|_{L^2}$")
    axes[0].legend(frameon=False)
    axes[0].set_yscale("log")
    axes[1].set_title("Free-energy error")
    axes[1].set_xlabel("time")
    axes[1].set_ylabel(r"$\|\widehat F - F_{TI}\|_{L^2}$")
    axes[1].legend(frameon=False)
    axes[1].set_yscale("log")
    plt.tight_layout()
    plt.show()


if results is not None and ti_reference is not None:
    attach_reference_errors(results, ti_reference)
    for method in METHOD_ORDER:
        c, tr, s = results[method]["state_fractions"][-1]
        extra = ""
        if method in {"wfr_abf", "kwfr_abf"}:
            extra = f", replacements={results[method]['total_replacement_events']}"
        print(f"{method:9s}: final marginal L2={results[method]['l2_marginal'][-1]:.4f}, sampled L2={results[method]['sampled_l2_marginal'][-1]:.4f}, C/T/S={c:.2f}/{tr:.2f}/{s:.2f}, L2 F'={results[method]['l2_mean_force_ref'][-1]:.3f}, L2 F={results[method]['l2_free_energy_ref'][-1]:.3f}{extra}")
    for method in METHOD_ORDER:
        plot_method_diagnostics(results, method, ti_reference, sim)
    plot_l2_errors(results)
else:
    print("Run both main simulation and TI reference before plotting diagnostics.")
